In [1]:
import json
import sys

In [2]:
PROJECT_DIR = '..'
sys.path.append('{}/code'.format(PROJECT_DIR))

In [3]:
from tb_eval import agents_2_max_completion_tokens, get_aoai_message_for_tbfactextraction
from tb_eval import get_aoai_message_for_tbfactcontextentailclassifier
from tb_eval import convert_to_tbgradedrubric, get_tbfact_score

from api_wrappers import get_aoai

/opt/anaconda3/envs/babelbench/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import yaml
def load_config(config_file_path):
    try:
        with open(config_file_path, 'r') as file:
            config = yaml.safe_load(file)
        return config
    except FileNotFoundError:
        print(f"Error: Configuration file not found at {config_file_path}")
        return None

In [5]:
tb_eval_prompts = load_config('../code/prompts/evals.yaml')

In [6]:
summary_system = 'Patient is a 68-year-old male diagnosed on 01.10.2025 with a glioblastoma in the right temporal lobe. Patient completed chemotherapy cycle for their multicentric lesion on 03.17.2020. Patient is currently receiving treatment with  gemcitabine and wishes to determine alternative treatment options towards treating their multifocal disease.'
summary_gold = 'Patient is a 65-year-old male diagnosed on 01.09.2020 with an MGMT methylated and IDH wildtype glioblastoma in the left temporal lobe. Patient completed their surgery and subtotal resection of their multicentric lesion on 01.09.2020 and their chemoradiation on 03.17.2020. Patient is currently receiving treatment with pembrolizumab and bevacizumab and wishes to determine alternative treatment options towards treating their multifocal disease.'

In [ ]:
#get facts for gold/system, classify, convert to tbgraderubric

In [7]:
agent = 'TBFactExtractor'
max_completion_tokens = agents_2_max_completion_tokens[agent]

In [8]:
messages = get_aoai_message_for_tbfactextraction( tb_eval_prompts, summary_gold )

In [9]:
gpt41_output = get_aoai( messages,
                                max_completion_tokens=max_completion_tokens,
                                deployment_name="gpt-4.1" )

In [10]:
facts_gold = json.loads( gpt41_output )

In [11]:
facts_gold

[{'text': '65-year-old male',
  'type': 'demographic',
  'mention': '65-year-old male',
  'attributes': []},
 {'text': 'diagnosed on 01.09.2020 with an MGMT methylated and IDH wildtype glioblastoma in the left temporal lobe',
  'type': 'diagnosis',
  'mention': 'glioblastoma',
  'attributes': ['left temporal lobe', '01.09.2020']},
 {'text': 'MGMT methylated',
  'type': 'testresult_molecular',
  'mention': 'MGMT methylated',
  'attributes': []},
 {'text': 'IDH wildtype',
  'type': 'testresult_pathology',
  'mention': 'IDH wildtype',
  'attributes': []},
 {'text': 'surgery and subtotal resection of their multicentric lesion on 01.09.2020',
  'type': 'treatment_surgery',
  'mention': 'surgery and subtotal resection',
  'attributes': ['multicentric lesion', '01.09.2020']},
 {'text': 'chemoradiation on 03.17.2020',
  'type': 'treatment_systemic',
  'mention': 'chemoradiation',
  'attributes': ['03.17.2020']},
 {'text': 'currently receiving treatment with pembrolizumab and bevacizumab',
  't

In [12]:
agent = 'TBFactContextEntailmentClassifier'
max_completion_tokens = agents_2_max_completion_tokens[agent]

In [13]:
messages = get_aoai_message_for_tbfactcontextentailclassifier( tb_eval_prompts, summary_gold, json.dumps(facts_gold), summary_system )

In [14]:
gpt41_output = get_aoai( messages,
                                max_completion_tokens=max_completion_tokens,
                                deployment_name="gpt-4.1" )

In [15]:
facts_gold_classified = json.loads( gpt41_output )

In [16]:
tbgradedrubric_gold = convert_to_tbgradedrubric( facts_gold_classified, candidate_text=summary_gold, reference_text=summary_system )

In [17]:
tbgradedrubric_gold.score_directional( additional_weights=None, error_weight=0.0 )

0.125

In [ ]:
#get facts for system, classify, convert to tbgraderubric

In [18]:
agent = 'TBFactExtractor'
max_completion_tokens = agents_2_max_completion_tokens[agent]

In [19]:
messages = get_aoai_message_for_tbfactextraction( tb_eval_prompts, summary_system )

In [20]:
gpt41_output = get_aoai( messages,
                                max_completion_tokens=max_completion_tokens,
                                deployment_name="gpt-4.1" )

In [21]:
facts_system = json.loads( gpt41_output )

In [22]:
agent = 'TBFactContextEntailmentClassifier'
max_completion_tokens = agents_2_max_completion_tokens[agent]

In [23]:
messages = get_aoai_message_for_tbfactcontextentailclassifier( tb_eval_prompts, summary_system, json.dumps(facts_system), summary_gold )

In [24]:
gpt41_output = get_aoai( messages,
                                max_completion_tokens=max_completion_tokens,
                                deployment_name="gpt-4.1" )

In [25]:
facts_system_classified = json.loads( gpt41_output )

In [26]:
tbgradedrubric_system = convert_to_tbgradedrubric( facts_system_classified, candidate_text=summary_system, reference_text=summary_gold )

In [27]:
tbgradedrubric_system.score_directional( additional_weights=None, error_weight=0.0 )

0.2

In [ ]:
#calculate scores

In [28]:
tbfact = get_tbfact_score( tbgradedrubric_gold, tbgradedrubric_system, error_weight=0)

In [29]:
tbfact

0.15384615384615385

In [30]:
from tb_eval import get_aoai_message_for_rating

In [31]:
agent = 'LLMAsJudge4Axes'
max_completion_tokens = agents_2_max_completion_tokens[agent]

In [32]:
messages = get_aoai_message_for_rating( tb_eval_prompts, summary_system, summary_gold, agent=agent)

In [33]:
messages

[{'role': 'system', 'content': 'You are a helpful medical assistant.\n'},
 {'role': 'user',
  'content': 'Given a candidate text {CANDIDATE TEXT} and a reference text {REFERENCE TEXT}, rate the {CANDIDATE} text on a scale of 1-5 (1 is lowest, 5 is best) for the categories described below. The judgement should be based on the content of the {REFERENCE TEXT} as the ground truth. The final {OUTPUT} should be in the form of a json object with the following keys and categories defined below.\n\nCategories:\n1. completeness: (Completeness) Using the {REFERENCE TEXT} as the ground truth, is all necessary information required for the case summary present in the {CANDIDATE TEXT}? (1 - contains no relevant/helpful information, 3 - Some critical elements present but some missing, 5 - All necessary information present)\n2. factual_accuracy: (Factual accuracy) Using the {REFERENCE TEXT} as the ground truth, is all content in the {CANDIDATE TEXT} medically accurate (regardless of if it is complete o

In [34]:
gpt41_output = get_aoai( messages,
                                max_completion_tokens=max_completion_tokens,
                                deployment_name="gpt-4.1" )

In [35]:
print( gpt41_output )

{
  "completeness": 3,
  "factual_accuracy": 1,
  "relevance": 1,
  "overall": 1
}

Explanation:
- Completeness: Some critical elements are present such as diagnosis, treatment history, and current decision-making, but key details (e.g., biomarker status, accurate dates, surgery details, exact treatments) are missing.
- Factual Accuracy: Multiple factual inaccuracies exist (different age, diagnosis date, tumor location, treatment agents, omission of surgery and biomarkers).
- Relevance: Most information is either inaccurate or not relevant to the reference patient; would not be helpful for clinical use.
- Overall: Major errors and revisions would be required to make this useful or safe for clinical decision-making.


In [36]:
agent = 'LLMAsJudgeGeneric'
max_completion_tokens = agents_2_max_completion_tokens[agent]

In [37]:
messages = get_aoai_message_for_rating( tb_eval_prompts, summary_system, summary_gold, agent=agent)

In [38]:
gpt41_output = get_aoai( messages,
                                max_completion_tokens=max_completion_tokens,
                                deployment_name="gpt-4.1" )

In [39]:
print( gpt41_output )

rating_1

Explanation: The candidate and reference texts overlap somewhat in topic (glioblastoma, treatment history, interest in alternative treatment options), but there are significant conflicting pieces of information:
- Age: 68 vs 65
- Date of diagnosis: 01.10.2025 vs 01.09.2020
- Location of tumor: right temporal lobe vs left temporal lobe
- Tumor characteristics: candidate omits MGMT methylated, IDH wildtype; reference includes them
- Treatments: candidate says completed chemotherapy cycle; reference says surgery, subtotal resection, chemoradiation
- Current treatment: candidate says gemcitabine; reference says pembrolizumab and bevacizumab
- Overlap: both mention multicentric/multifocal disease and the wish for alternative treatment options

Given the conflicting details about diagnosis, age, location, treatment history, and current treatment, this fits rating_1.
